# पाठ 11 - एजेंट-टू-एजेंट (A2A) प्रोटोकॉल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A प्रोटोकॉल क्या है?

यह **Agent-to-Agent (A2A) प्रोटोकॉल** एक खुला मानक है जो AI एजेंट्स को संवाद करने,
एक-दूसरे को खोजने, और सहयोग करने — भले ही वे विभिन्न फ्रेमवर्क पर बने हों या
विभिन्न सेवाओं द्वारा होस्ट किए गए हों।

मुख्य अवधारणाएँ:

- **Discovery** – एजेंट्स एक *एजेंट कार्ड* प्रकाशित करते हैं जो उनकी क्षमताओं का वर्णन करता है, जिससे
  अन्य एजेंट्स (या समन्वयक) के लिए किसी कार्य के लिए सही विशेषज्ञ को ढूँढना आसान हो जाता है।
- **Message Passing** – एजेंट्स एक सामान्य प्रोटोकॉल के माध्यम से संरचित संदेशों का आदान-प्रदान करते हैं, ताकि एक
  एजेंट द्वारा किया गया अनुरोध दूसरे द्वारा उसकी आंतरिक
  कार्यान्वयन द्वारा समझा और पूरा किया जा सके।
- **Task Lifecycle** – A2A उन अवस्थाओं को परिभाषित करता है जैसे *जमा किया गया*, *कार्यरत*, *पूर्ण*, और
  *विफल*, जिससे समन्वयक को यह पूरी दृश्यता मिलती है कि सौंपा गया कार्य कैसे प्रगति कर रहा है।

इस पाठ में हम A2A-शैली सहयोग का अनुकरण करते हैं, तीन विशेषीकृत यात्रा एजेंटों को
एक वर्कफ़्लो में जोड़कर जहाँ प्रत्येक एजेंट अपनी विशेषज्ञता योगदान देता है और परिणाम अगले को सौंपता है।


## विशेषीकृत यात्रा एजेंट बनाना


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## वर्कफ़्लो के माध्यम से बहु-एजेंट सहयोग

हम तीन एजेंटों को एक क्रमिक वर्कफ़्लो में जोड़ते हैं जो A2A संदेश पासिंग को दर्शाता है:

1. **CurrencyExchangeAgent** उपयोगकर्ता अनुरोध प्राप्त करता है और मुद्रा मार्गदर्शन प्रदान करता है.
2. **ActivityPlannerAgent** समृद्ध संदर्भ प्राप्त करता है और गतिविधि सिफारिशें जोड़ता है.
3. **TravelManagerAgent** दोनों इनपुटों को समेकित करके एक अंतिम यात्रा सारांश तैयार करता है.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## प्रोडक्शन में A2A को समझना

प्रोडक्शन वातावरण में A2A प्रोटोकॉल शक्तिशाली क्रॉस-सर्विस परिदृश्यों को सक्षम करता है:

| क्षमता | विवरण |
|---|---|
| **क्रॉस-फ्रेमवर्क इंटरऑप** | एक फ्रेमवर्क के साथ बनाए गए एजेंट किसी भी अन्य A2A-अनुपालन फ्रेमवर्क के साथ बनाए गए एजेंट को कार्य सौंप सकते हैं, जिससे वास्तविक क्रॉस-ऑर्गनाइज़ेशन इंटरऑपरेबिलिटी संभव होती है। |
| **सेवा सीमाएँ** | एजेंट अलग माइक्रोसर्विसेज़, क्लाउड क्षेत्रों, या यहां तक कि अलग संगठनों में रह सकते हैं और फिर भी निर्बाध रूप से सहयोग कर सकते हैं। |
| **डायनामिक खोज** | एक ऑर्केस्ट्रेटर रनटाइम में एक Agent Card रजिस्ट्री से क्वेरी कर सकता है ताकि किसी दिए गए उप-कार्य के लिए सबसे उपयुक्त विशेषज्ञ मिल सके। |
| **स्ट्रीमिंग और पुश नोटिफिकेशन्स** | A2A Server-Sent Events (SSE) का समर्थन करता है जो रीयल-टाइम प्रोग्रेस अपडेट और लंबी अवधि के कार्यों के लिए पुश नोटिफिकेशन्स प्रदान करता है। |

उपर्युक्त वर्कफ़्लो जो हमने बनाया वह इस पैटर्न का एक सरल, इन-प्रोसेस संस्करण है। वास्तविक
परिनियोजन में प्रत्येक एजेंट एक HTTP endpoint एक्सपोज़ करेगा, एक Agent Card प्रकाशित करेगा, और
A2A JSON-RPC प्रोटोकॉल के माध्यम से संवाद करेगा।


## सारांश

इस पाठ में आपने सीखा:

1. **A2A प्रोटोकॉल क्या है** — एजेंट-से-एजेंट खोज, संदेश,
   और कार्य प्रबंधन.
2. **विशेषीकृत एजेंट कैसे बनाएँ** — एक मुद्रा विनिमय एजेंट, एक गतिविधि योजनाकार एजेंट,
   और एक यात्रा प्रबंधक समन्वयक.
3. **एजेंट्स को एक वर्कफ़्लो में जोड़ने का तरीका** — `WorkflowBuilder` का उपयोग करके अनुक्रमिक
   संदेश पासिंग को एजेंटों के बीच मॉडल करना।
4. **A2A उत्पादन में कैसे काम करता है** — क्रॉस-फ्रेमवर्क, क्रॉस-सर्विस सहयोग को सक्षम करना
   गतिशील खोज और स्ट्रीमिंग अपडेट्स के साथ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा Co-op Translator (https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। हम सटीकता के लिए प्रयास करते हैं, फिर भी कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ को उसकी मूल भाषा में आधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
